![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 4 — Lab 2: Sentiment Analysis — From Scratch vs. Pretrained

**Contributed by Sattam Altwaim**

---

Is a movie review **positive** or **negative**?

We will solve this same task **two completely different ways** and compare them at the end:

| Approach | What we build | What it learns from |
|---|---|---|
| **Part 1** | BPE tokenizer + LSTM | Only our small dataset |
| **Part 2** | Frozen DistilBERT + classifier | Billions of words of English text |

By the end you will see concretely **why pretrained models usually win** when data is limited — and understand exactly what each step is doing.

---
## Roadmap

```
Part 1 — From Scratch
  Step 1  Install & import
  Step 2  Load the dataset
  Step 3  Train a BPE tokenizer
  Step 4  Encode text into number sequences
  Step 5  Build an LSTM model
  Step 6  Train the LSTM
  Step 7  Evaluate it

Part 2 — Pretrained DistilBERT
  Step 8   Load frozen DistilBERT
  Step 9   Extract BERT embeddings
  Step 10  Train a logistic classifier on embeddings
  Step 11  Compare both approaches
```


---
## Step 1 — Install & Import

We need three libraries beyond the standard PyTorch stack:
- **`datasets`** — HuggingFace's library for loading standard NLP datasets in one line
- **`transformers`** — provides pretrained models (BERT, GPT, etc.) and their tokenizers
- **`tokenizers`** — fast Rust-backed library for building custom tokenizers (we use it in Part 1)

In [ ]:
%pip install datasets transformers tokenizers scikit-learn -q

import torch
import torch.nn as nn
import os, warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

from datasets import load_dataset
import datasets; datasets.logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
## Step 2 — Load the Dataset

We use the **Rotten Tomatoes** movie review dataset:
- Short one-sentence snippets from real movie reviews
- Each is labelled **positive (1)** or **negative (0)**
- ~8,500 training reviews, ~1,000 test reviews — a deliberately small dataset

> **Why small?** We want to show the real-world situation where you don't have millions of labelled examples. This is where pretrained models shine the most.


In [ ]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

train_texts  = data["train"]["text"]
train_labels = data["train"]["label"]
test_texts   = data["test"]["text"]
test_labels  = data["test"]["label"]

print(f"Training reviews: {len(train_texts)}")
print(f"Test reviews:     {len(test_texts)}")
print()
print("Sample positive review:", train_texts[0])
print("Label:", train_labels[0], "→ positive" if train_labels[0] == 1 else "→ negative")
print()
print("Sample negative review:", train_texts[-1])
print("Label:", train_labels[-1], "→ positive" if train_labels[-1] == 1 else "→ negative")

---
# Part 1 — From Scratch: BPE Tokenizer + LSTM

In this part we build everything ourselves — no pretrained weights, no borrowed knowledge.
The model has to learn English sentiment purely from our ~8,500 training reviews.

The pipeline looks like this:

```
Raw text
    ↓  [BPE Tokenizer]  — splits words into subword pieces, maps each to a number
Token IDs (padded to fixed length)
    ↓  [Embedding Layer]  — maps each token number to a learned vector
Sequence of vectors
    ↓  [LSTM]  — reads the sequence left-to-right, summarises into one vector
Single summary vector
    ↓  [Linear layer]  — maps the summary to 2 scores (positive / negative)
Prediction
```


---
## Step 3 — Train a BPE Tokenizer

### What is BPE?

**Byte Pair Encoding (BPE)** is a subword tokenization algorithm.
Instead of splitting text into whole words (which would make our vocabulary huge and miss typos/rare words),
BPE finds the most common *pairs of characters or character sequences* and merges them into one token.

**Example:** the word `"wonderful"` might become `["wonder", "##ful"]`

BPE starts from individual characters and iteratively merges the most frequent pair:
```
s c i e n c e  →  sc i en ce  →  sci en ce  →  scie nce  →  science
```

**Why BPE over whole-word tokenization?**
- Handles unknown or rare words — `"unbelievably"` = `["un", "believ", "ably"]`
- Vocabulary stays manageable (we use 5,000 tokens here)
- Same approach used inside GPT-2, GPT-3, and GPT-4

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Build a BPE tokenizer and train it on our movie reviews
tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()   # first split on spaces, then apply BPE

trainer = trainers.BpeTrainer(
    vocab_size=5000,
    special_tokens=["[PAD]", "[UNK]"]  # [PAD] = padding, [UNK] = unknown token
)

tok.train_from_iterator(train_texts, trainer)
print(f"Vocabulary size: {tok.get_vocab_size()}")

# Let's see what BPE does to a sample sentence
sample = "This movie was absolutely wonderful!"
encoded = tok.encode(sample)
print(f"\nSample text: {sample}")
print(f"Tokens: {encoded.tokens}")
print(f"Token IDs: {encoded.ids}")

---
## Step 4 — Encode Text into Fixed-Length Number Sequences

Neural networks need tensors — fixed-shape grids of numbers.
But reviews are different lengths. We solve this two ways:
- **Truncate:** keep at most the first 40 tokens (long reviews get cut)
- **Pad:** add `[PAD]` tokens to the end of short reviews so they all reach length 40

```
"Great film"   → [12, 450, 0, 0, 0, 0, ...]   ← padded to length 40
"Terrible and a total waste of time and money"  → [33, 87, 4, 19, ...] ← truncated to 40
```

We chose **40 tokens** because Rotten Tomatoes reviews are short snippets — 40 is enough to capture the full meaning without wasting compute on padding.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

PAD    = tok.token_to_id("[PAD]")
MAXLEN = 40

def encode(texts):
    '''Convert a list of strings into a padded/truncated tensor of token IDs.'''
    seqs = []
    for t in texts:
        ids = tok.encode(t).ids[:MAXLEN]       # truncate to MAXLEN
        ids += [PAD] * (MAXLEN - len(ids))     # pad short reviews
        seqs.append(ids)
    return torch.tensor(seqs)

Xtr = encode(train_texts)
ytr = torch.tensor(train_labels)
Xte = encode(test_texts)
yte = torch.tensor(test_labels)

print(f"Training tensor shape: {Xtr.shape}  (num_reviews × max_len)")
print(f"Labels shape:          {ytr.shape}")
print(f"\nFirst review encoded:\n{Xtr[0]}")
print(f"\nOriginal text: {train_texts[0]}")

---
## Step 5 — Build the LSTM Model

### What is an LSTM?

An **LSTM (Long Short-Term Memory)** is a type of recurrent neural network designed to process *sequences*.
Unlike a regular neural network that processes everything at once, an LSTM reads tokens **one at a time**, left to right,
carrying a hidden state (a memory vector) that gets updated at each step.

```
Token 1 → LSTM → hidden state 1
Token 2 → LSTM → hidden state 2
Token 3 → LSTM → hidden state 3   ← final hidden state summarizes the whole review
...
```

The **final hidden state** is a single vector that (ideally) captures the overall meaning/sentiment of the entire review.

### Our architecture:
```
[PAD-aware Embedding]  vocab_size=5000, dim=64
         ↓
      [LSTM]           input=64, hidden=128
         ↓
 [Final hidden state]  shape: (128,)
         ↓
  [Linear classifier]  128 → 2  (negative / positive)
```

**Note on `padding_idx`:** The `Embedding` layer skips updating the `[PAD]` token's vector during training,
since padding is not meaningful text.

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        # Embedding: maps each token ID to a learned vector of size embed_dim
        # padding_idx tells it to ignore [PAD] tokens when computing gradients
        self.emb  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)

        # LSTM: reads the sequence of vectors, outputs a hidden state per step
        # batch_first=True means input shape is (batch, seq_len, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Final linear layer: maps the last hidden state to 2 class scores
        self.fc   = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        # x shape: (batch, seq_len)
        embedded = self.emb(x)                    # → (batch, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(embedded)         # h_n: (1, batch, hidden_dim)
        return self.fc(h_n[-1])                   # → (batch, 2)

lstm_model = SentimentLSTM(tok.get_vocab_size())

# Count parameters
total_params = sum(p.numel() for p in lstm_model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"\nModel architecture:\n{lstm_model}")

---
## Step 6 — Train the LSTM

Standard PyTorch training loop:
1. **Forward pass** — feed a batch of reviews through the model, get predictions
2. **Loss** — measure how wrong the predictions are (CrossEntropyLoss)
3. **Backward pass** — compute gradients (which direction to nudge each weight)
4. **Optimizer step** — nudge all weights by a small amount in the right direction

We use **Adam** optimizer with learning rate `1e-3` — a reliable default for most sequence models.

> Each *epoch* = one full pass over all 8,500 training reviews.
> With batch size 64, that's ~133 batches per epoch.

In [ ]:
lstm_model = SentimentLSTM(tok.get_vocab_size())
lstm_model.train()

train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=64, shuffle=True
)

optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

train_losses = []

for epoch in range(5):
    epoch_loss = 0.0
    num_batches = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = lstm_model(xb)
        loss  = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss  += loss.item()
        num_batches += 1
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5  |  Loss: {avg_loss:.4f}")

print("\nTraining complete.")

---
## Step 7 — Evaluate the LSTM

We switch the model to **eval mode** (`model.eval()`) which:
- Disables dropout (if any) so predictions are deterministic
- Disables gradient tracking (`torch.no_grad()`) to save memory and speed up inference

We then run the entire test set through and compare predictions to true labels.

In [ ]:
import matplotlib.pyplot as plt

lstm_model.eval()
with torch.no_grad():
    test_preds = lstm_model(Xte).argmax(dim=1)
    lstm_acc = (test_preds == yte).float().mean().item()

print(f"LSTM Test Accuracy: {lstm_acc:.1%}")

# Plot training loss curve
plt.figure(figsize=(7, 3))
plt.plot(range(1, 6), train_losses, 'b-o')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('LSTM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 7b — Try Your Own Sentence with the LSTM

In [ ]:
def predict_lstm(text):
    '''Predict sentiment for a single text string using the trained LSTM.'''
    lstm_model.eval()
    with torch.no_grad():
        tokens = encode([text])     # shape: (1, 40)
        pred   = lstm_model(tokens).argmax(dim=1).item()
    return "✅ positive" if pred == 1 else "❌ negative"

print(predict_lstm("An absolute masterpiece, I loved every minute."))
print(predict_lstm("Boring, predictable and a total waste of time."))
print(predict_lstm("A surprisingly moving story with great performances."))
print(predict_lstm("I've seen better acting in a school play."))

---
# Part 2 — Pretrained: Frozen DistilBERT + Classifier

In Part 1, the model started with **zero knowledge** of English. It had to learn everything from our 8,500 reviews.

In Part 2, we use **DistilBERT** — a model pre-trained on billions of words of text (Wikipedia + BooksCorpus).
It already knows English grammar, sentiment cues, word relationships, and nuance.

We will:
1. **Freeze** DistilBERT (don't update its weights — just use them as-is)
2. Run all reviews through it to get rich **embedding vectors**
3. Train a simple **logistic regression classifier** on those embeddings

```
Raw text
    ↓  [DistilBERT Tokenizer]
Token IDs
    ↓  [Frozen DistilBERT]  ← 66M pretrained parameters, not updated
Per-token vectors (averaged)
    ↓  [Logistic Regression]  ← only this tiny classifier trains
Prediction
```

> **Why DistilBERT and not full BERT?**
> DistilBERT is 40% smaller and 60% faster than BERT while keeping 97% of its performance —
> a great choice for teaching and fast experimentation.


---
## Step 8 — Load the Frozen DistilBERT

We load:
- **`AutoTokenizer`** — DistilBERT's own tokenizer (WordPiece, trained on its pretraining data)
- **`AutoModel`** — the DistilBERT model itself with all pretrained weights

We call `.eval()` to put it in inference mode and make sure gradients are never computed through it.

> **Note:** The correct model path is `"distilbert/distilbert-base-uncased"`. The old alias `"distilbert-base-uncased"` can fail on newer versions of `transformers`.

In [ ]:
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# ── FIX: use the full organisation/model-name path ──
BERT_MODEL = "distilbert/distilbert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model     = AutoModel.from_pretrained(BERT_MODEL).to(device).eval()

bert_params = sum(p.numel() for p in bert_model.parameters())
print(f"DistilBERT parameters: {bert_params:,}  (all frozen — none will be updated)")
print(f"Running on: {device}")

---
## Step 9 — Extract DistilBERT Embeddings


In [ ]:
import numpy as np

@torch.no_grad()
def get_bert_embeddings(texts, batch_size=64):
    '''
    Pass texts through frozen DistilBERT, return mean-pooled embeddings.
    Returns a numpy array of shape (len(texts), 768).
    '''
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        # Tokenize: truncate to 40 tokens, pad shorter sequences, return PyTorch tensors
        encoded = bert_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=40,
            return_tensors="pt"
        ).to(device)

        # Run through DistilBERT → one vector per token per review
        outputs = bert_model(**encoded)
        hidden  = outputs.last_hidden_state    # shape: (batch, seq_len, 768)

        # Average over real tokens only (mask out [PAD] positions)
        mask = encoded["attention_mask"].unsqueeze(-1).float()  # (batch, seq_len, 1)
        mean_pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (batch, 768)

        all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

print("Extracting training embeddings...")
train_emb = get_bert_embeddings(train_texts)
print("Extracting test embeddings...")
test_emb  = get_bert_embeddings(test_texts)

print(f"\nTraining embeddings shape: {train_emb.shape}  (num_reviews × 768)")
print(f"Test embeddings shape:     {test_emb.shape}")

---
## Step 10 — Train a Logistic Regression Classifier

Now that every review is a 768-dimensional vector, we train a simple **logistic regression** on top.

Logistic regression learns a linear boundary in the 768-dimensional space:
- Reviews on one side → positive
- Reviews on the other side → negative

This is tiny — just 768 × 2 + 2 = 1,538 parameters to learn.
But because the BERT embeddings are already rich and meaningful, this simple linear classifier can do very well.

> This is the same concept as in the transfer learning lab — **freeze the backbone, train only the head** —
> except here the "head" is a logistic regression instead of a neural network layer.

In [ ]:
from sklearn.linear_model import LogisticRegression

print("Training logistic regression classifier on BERT embeddings...")
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(train_emb, train_labels)

bert_acc = clf.score(test_emb, test_labels)
print(f"\nBERT + Logistic Regression Test Accuracy: {bert_acc:.1%}")

---
## Step 11 — Compare: LSTM vs. Pretrained BERT

In [ ]:
print("=" * 50)
print(f"  From-scratch LSTM:    {lstm_acc:.1%}")
print(f"  Pretrained BERT:      {bert_acc:.1%}")
print("=" * 50)
print(f"  Improvement:          +{(bert_acc - lstm_acc):.1%}")
print()

# Plot comparison bar chart
import matplotlib.pyplot as plt

methods    = ['From-scratch\nLSTM', 'Pretrained\nDistilBERT']
accuracies = [lstm_acc * 100, bert_acc * 100]
colors     = ['#7890B4', '#E07A3C']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(methods, accuracies, color=colors, edgecolor='white', linewidth=1.2, width=0.4)
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("LSTM vs. Pretrained BERT\n(Rotten Tomatoes Sentiment)")
ax.set_ylim(50, 100)
ax.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

---
## Step 12 — Try Your Own Sentence with BERT

In [ ]:
def predict_bert(text):
    '''Predict sentiment for a single text string using the BERT + LR pipeline.'''
    emb  = get_bert_embeddings([text])   # shape: (1, 768)
    pred = clf.predict(emb)[0]
    prob = clf.predict_proba(emb)[0]
    label = "✅ positive" if pred == 1 else "❌ negative"
    confidence = max(prob) * 100
    return f"{label}  (confidence: {confidence:.1f}%)"

print(predict_bert("An absolute masterpiece, I loved every minute."))
print(predict_bert("Boring, predictable and a total waste of time."))
print(predict_bert("A surprisingly moving story with great performances."))
print(predict_bert("I've seen better acting in a school play."))

---
## Takeaway

| | LSTM (from scratch) | DistilBERT (pretrained) |
|---|---|---|
| **What it knows before training** | Nothing | English from billions of words |
| **Parameters trained on our data** | ~700K (all of them) | ~1,500 (just the classifier) |
| **Training time** | ~1 min (5 epochs) | Seconds (just logistic regression) |
| **Typical accuracy** | ~75–80% | ~85–90% |

**The core lesson:** with only a few thousand labelled examples, a frozen pretrained model almost always
outperforms a model trained from scratch — because the pretrained model already *understands language*.
You are only teaching it the last step: "in this context, which class is this?"

This is **transfer learning** applied to NLP, and it's the reason models like BERT, GPT, and LLaMA
have transformed the field — you can solve real NLP tasks with very little labelled data.

---
*Lab contributed by Sattam Altwaim & Hussain Alyafei*
